# Web Security Scanner
**Project topic:** Automated Web Scanner: programmed a Python script to audit sites for secure headers and CORS flaws.

**What it does:** sends safe, read-only requests to a web application and checks
1. HTTP security headers (CSP, clickjacking protection, X-Content-Type-Options, Referrer-Policy, Permissions-Policy, HSTS),
2. cookie flags (HttpOnly, Secure, SameSite),
3. information leaks in the `Server` and `X-Powered-By` headers,
4. CORS configuration (reflected origins, wildcard, `null` origin, risky preflight methods).

Each finding gets a severity (High / Medium / Low), a fix recommendation, and each site gets a score out of 100 and a grade (A-F).

**Test targets:** so the results are reproducible and legal to run anywhere, the notebook starts three small local web servers with different configurations (insecure, partially secured, hardened) and scans them. You can also add your own sites in section 5.

> **Use responsibly:** only scan sites you own or have written permission to test.

In [ ]:
import json, os, re, threading
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
sns.set_theme(style="whitegrid")
print("requests", requests.__version__)

## 1. Local test servers
Three tiny servers, each configured differently on purpose. They give the scanner known answers to be checked against.

| Server | Intended configuration |
|---|---|
| `insecure-lab` | No security headers, version-leaking headers, bare cookie, CORS reflects any origin with credentials |
| `partial-lab` | Some headers set, wildcard CORS, cookie missing SameSite |
| `hardened-lab` | Full set of headers, strict CSP, secure cookie, CORS limited to one trusted origin |

In [ ]:
TRUSTED_ORIGIN = "https://app.example.com"

class LabHandler(BaseHTTPRequestHandler):
    profile = {}
    def version_string(self):            # value of the Server header
        return self.profile["server"]
    def log_message(self, *args):        # keep the notebook output clean
        pass
    def _respond(self, body):
        p = self.profile
        self.send_response(200)
        self.send_header("Content-Type", "text/html; charset=utf-8")
        for k, v in p["headers"].items():
            self.send_header(k, v)
        for c in p["cookies"]:
            self.send_header("Set-Cookie", c)
        for k, v in p["cors"](self.headers.get("Origin")).items():
            self.send_header(k, v)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        if self.command != "HEAD":
            self.wfile.write(body)
    def do_GET(self):     self._respond(b"<html><body>lab page</body></html>")
    def do_HEAD(self):    self._respond(b"")
    def do_OPTIONS(self): self._respond(b"")

PROFILES = {
    "insecure-lab": {
        "server": "Apache/2.2.15 (CentOS)",
        "headers": {"X-Powered-By": "PHP/5.3.3"},
        "cookies": ["PHPSESSID=abc123; Path=/"],
        "cors": lambda o: ({"Access-Control-Allow-Origin": o, "Access-Control-Allow-Credentials": "true",
                            "Access-Control-Allow-Methods": "GET, POST, PUT, DELETE"} if o else {}),
    },
    "partial-lab": {
        "server": "nginx",
        "headers": {"X-Content-Type-Options": "nosniff", "X-Frame-Options": "SAMEORIGIN"},
        "cookies": ["session=abc123; Path=/; HttpOnly"],
        "cors": lambda o: {"Access-Control-Allow-Origin": "*"},
    },
    "hardened-lab": {
        "server": "webserver",
        "headers": {
            "Content-Security-Policy": "default-src 'self'; frame-ancestors 'none'; object-src 'none'",
            "X-Frame-Options": "DENY",
            "X-Content-Type-Options": "nosniff",
            "Referrer-Policy": "strict-origin-when-cross-origin",
            "Permissions-Policy": "geolocation=(), camera=(), microphone=()",
            "Strict-Transport-Security": "max-age=31536000; includeSubDomains",
        },
        "cookies": ["session=abc123; Path=/; HttpOnly; Secure; SameSite=Strict"],
        "cors": lambda o: ({"Access-Control-Allow-Origin": o, "Access-Control-Allow-Methods": "GET",
                            "Vary": "Origin"} if o == TRUSTED_ORIGIN else {"Vary": "Origin"}),
    },
}

servers, TARGETS = [], {}
for name, prof in PROFILES.items():
    handler = type(name.replace("-", "_"), (LabHandler,), {"profile": prof})
    srv = HTTPServer(("127.0.0.1", 0), handler)          # port 0 = any free port
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    servers.append(srv)
    TARGETS[name] = f"http://127.0.0.1:{srv.server_address[1]}/"
TARGETS

## 2. The scanner
`scan_target()` runs every check and returns a list of results. Every check is recorded, including the ones that pass, so the final report can show a full pass/fail matrix.

Scoring: start at 100 and subtract 20 for each High finding, 10 for Medium and 4 for Low. Grades: A >= 90, B >= 80, C >= 65, D >= 50, F below 50.

In [ ]:
PENALTY = {"High": 20, "Medium": 10, "Low": 4}
EVIL_ORIGIN = "https://evil.example"

def is_local(host):
    return host in ("localhost", "127.0.0.1", "::1")

def http(url, method="GET", headers=None):
    return requests.request(method, url, headers=headers or {}, timeout=8, allow_redirects=False)

def scan_target(name, url):
    results = []
    def add(category, check, severity, detail, fix=""):
        results.append({"target": name, "category": category, "check": check,
                        "severity": severity, "detail": detail, "fix": fix})

    parsed = urlparse(url)
    is_https = parsed.scheme == "https"
    r = http(url)
    h = {k.lower(): v for k, v in r.headers.items()}

    # --- Transport ---------------------------------------------------------
    if is_https:
        if "strict-transport-security" not in h:
            add("Headers", "HSTS", "High", "Strict-Transport-Security header is missing.",
                "Add Strict-Transport-Security: max-age=31536000; includeSubDomains")
        else:
            add("Headers", "HSTS", "Pass", h["strict-transport-security"])
    elif is_local(parsed.hostname):
        add("Headers", "HSTS", "N/A", "Plain HTTP on localhost; HSTS only applies to HTTPS.")
    else:
        add("Headers", "HSTS", "High", "Site is served over plain HTTP, so HSTS and secure cookies cannot work.",
            "Serve the site over HTTPS and redirect all HTTP traffic.")

    # --- Content-Security-Policy ------------------------------------------
    csp = h.get("content-security-policy")
    if not csp:
        add("Headers", "Content-Security-Policy", "Medium", "No CSP header; the browser will run any injected script.",
            "Define a CSP such as default-src 'self'.")
    elif re.search(r"unsafe-inline|unsafe-eval|(^|[\s;])\*([\s;]|$)", csp):
        add("Headers", "Content-Security-Policy", "Medium", f"CSP is present but weak: {csp}",
            "Remove 'unsafe-inline', 'unsafe-eval' and wildcard sources; use nonces or hashes.")
    else:
        add("Headers", "Content-Security-Policy", "Pass", csp)

    # --- Clickjacking -----------------------------------------------------
    xfo = h.get("x-frame-options", "").upper()
    if xfo in ("DENY", "SAMEORIGIN") or (csp and "frame-ancestors" in csp):
        add("Headers", "Clickjacking protection", "Pass", xfo or "CSP frame-ancestors")
    else:
        add("Headers", "Clickjacking protection", "Medium", "Neither X-Frame-Options nor CSP frame-ancestors is set.",
            "Add X-Frame-Options: DENY or CSP frame-ancestors 'none'.")

    # --- Simple presence checks -------------------------------------------
    for hdr, sev, want, fix in [
        ("x-content-type-options", "Low", "nosniff", "Add X-Content-Type-Options: nosniff"),
        ("referrer-policy", "Low", None, "Add Referrer-Policy: strict-origin-when-cross-origin"),
        ("permissions-policy", "Low", None, "Add a Permissions-Policy that disables unused features."),
    ]:
        label = "-".join(w.capitalize() for w in hdr.split("-"))
        v = h.get(hdr)
        if v and (want is None or v.lower() == want):
            add("Headers", label, "Pass", v)
        else:
            add("Headers", label, sev, f"{label} header is missing or has an unexpected value.", fix)

    # --- Information leakage ----------------------------------------------
    srv = h.get("server", "")
    if re.search(r"\d", srv):
        add("Info leak", "Server version", "Low", f"Server header reveals version: {srv}",
            "Hide version details (for example server_tokens off).")
    else:
        add("Info leak", "Server version", "Pass", srv or "no Server header")
    if "x-powered-by" in h:
        add("Info leak", "X-Powered-By", "Low", f"X-Powered-By reveals technology: {h['x-powered-by']}",
            "Remove the X-Powered-By header.")
    else:
        add("Info leak", "X-Powered-By", "Pass", "header not present")

    # --- Cookies -----------------------------------------------------------
    cookies = r.raw.headers.getlist("Set-Cookie") if hasattr(r.raw.headers, "getlist") else []
    if not cookies:
        for chk in ("Cookie HttpOnly", "Cookie SameSite", "Cookie Secure"):
            add("Cookies", chk, "N/A", "No cookies set on this page.")
    else:
        attrs = [c.lower().replace(" ", "").split(";") for c in cookies]
        cname = cookies[0].split("=")[0]
        if all("httponly" in a for a in attrs):
            add("Cookies", "Cookie HttpOnly", "Pass", f"{cname} has HttpOnly")
        else:
            add("Cookies", "Cookie HttpOnly", "Medium", f"Cookie {cname} can be read by JavaScript.",
                "Add the HttpOnly flag.")
        if all(any(x.startswith("samesite=") for x in a) for a in attrs):
            add("Cookies", "Cookie SameSite", "Pass", f"{cname} has SameSite")
        else:
            add("Cookies", "Cookie SameSite", "Low", f"Cookie {cname} has no SameSite attribute.",
                "Add SameSite=Lax or Strict.")
        if not is_https and is_local(parsed.hostname):
            add("Cookies", "Cookie Secure", "N/A", "Plain HTTP on localhost.")
        elif all("secure" in a for a in attrs):
            add("Cookies", "Cookie Secure", "Pass", f"{cname} has Secure")
        else:
            add("Cookies", "Cookie Secure", "Medium", f"Cookie {cname} can be sent over plain HTTP.",
                "Add the Secure flag.")

    # --- CORS ---------------------------------------------------------------
    def cors_of(origin, method="GET", extra=None):
        hdrs = {"Origin": origin}
        hdrs.update(extra or {})
        resp = http(url, method, hdrs)
        g = {k.lower(): v for k, v in resp.headers.items()}
        return g.get("access-control-allow-origin"), g.get("access-control-allow-credentials", "").lower() == "true", g

    acao, creds, _ = cors_of(EVIL_ORIGIN)
    if acao == EVIL_ORIGIN:
        add("CORS", "Reflected origin", "High" if creds else "Medium",
            f"Server echoes any Origin back{' and allows credentials' if creds else ''}; any website can read responses.",
            "Allow only a fixed list of trusted origins.")
    elif acao == "*":
        add("CORS", "Reflected origin", "Pass", "Origin is not reflected (wildcard checked below).")
    else:
        add("CORS", "Reflected origin", "Pass", "Untrusted origin was not allowed.")

    if acao == "*":
        add("CORS", "Wildcard origin", "Medium" if creds else "Low",
            "Access-Control-Allow-Origin is * so every site can read this response.",
            "Replace * with specific trusted origins unless the content is fully public.")
    else:
        add("CORS", "Wildcard origin", "Pass", "no wildcard")

    acao_n, creds_n, _ = cors_of("null")
    if acao_n == "null":
        add("CORS", "Null origin", "High" if creds_n else "Medium",
            "The 'null' origin is trusted (sandboxed iframes and local files can use it).",
            "Never allow the null origin.")
    else:
        add("CORS", "Null origin", "Pass", "null origin not trusted")

    acao_p, _, gp = cors_of(EVIL_ORIGIN, "OPTIONS", {"Access-Control-Request-Method": "DELETE"})
    methods = gp.get("access-control-allow-methods", "")
    if acao_p in (EVIL_ORIGIN, "*") and re.search(r"PUT|DELETE|PATCH", methods, re.I):
        add("CORS", "Preflight methods", "Low", f"Untrusted origins may use risky methods: {methods}",
            "Limit Access-Control-Allow-Methods to what the API needs.")
    else:
        add("CORS", "Preflight methods", "Pass", "no risky methods allowed to untrusted origins")

    return results

def score_and_grade(df_target):
    penalty = sum(PENALTY.get(s, 0) for s in df_target["severity"])
    score = max(0, 100 - penalty)
    grade = "A" if score >= 90 else "B" if score >= 80 else "C" if score >= 65 else "D" if score >= 50 else "F"
    return score, grade

## 3. Run the scan

In [ ]:
all_results = []
for name, url in TARGETS.items():
    try:
        all_results += scan_target(name, url)
    except Exception as e:
        all_results.append({"target": name, "category": "Scan", "check": "Reachable", "severity": "High",
                            "detail": f"Could not scan: {type(e).__name__}", "fix": "Check the URL and network."})

findings = pd.DataFrame(all_results)
summary = []
for name, g in findings.groupby("target", sort=False):
    s, grade = score_and_grade(g)
    counts = g["severity"].value_counts()
    summary.append({"Target": name, "Score": s, "Grade": grade,
                    "High": counts.get("High", 0), "Medium": counts.get("Medium", 0),
                    "Low": counts.get("Low", 0), "Passed": counts.get("Pass", 0)})
summary = pd.DataFrame(summary).set_index("Target")

findings.to_csv(f"{RESULTS_DIR}/findings.csv", index=False)
with open(f"{RESULTS_DIR}/scan_results.json", "w") as f:
    json.dump({"summary": summary.reset_index().to_dict(orient="records"),
               "findings": findings.to_dict(orient="records")}, f, indent=2)
summary

In [ ]:
issues = findings[findings["severity"].isin(["High", "Medium", "Low"])].copy()
order = {"High": 0, "Medium": 1, "Low": 2}
issues = issues.sort_values(by=["target", "severity"], key=lambda c: c.map(order) if c.name == "severity" else c, kind="stable")
pd.set_option("display.max_colwidth", 90)
issues[["target", "category", "check", "severity", "detail"]].reset_index(drop=True)

## 4. Charts

In [ ]:
# Score per target
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#E45756" if s < 50 else "#F2B134" if s < 90 else "#54A24B" for s in summary["Score"]]
bars = ax.bar(summary.index, summary["Score"], color=colors)
for b, (idx, row) in zip(bars, summary.iterrows()):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1.5, f"{row['Score']} ({row['Grade']})", ha="center")
ax.set_ylim(0, 115); ax.set_ylabel("Score (out of 100)"); ax.set_title("Security score per target")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/scores.png", dpi=150); plt.show()

# Severity counts
summary[["High", "Medium", "Low"]].plot.bar(stacked=True, figsize=(7, 4), color=["#E45756", "#F2B134", "#4C78A8"])
plt.ylabel("Number of findings"); plt.title("Findings by severity"); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/severity.png", dpi=150); plt.show()

# Pass/fail matrix
pivot = findings.pivot_table(index="check", columns="target", values="severity", aggfunc="first")
pivot = pivot[list(TARGETS)]
rank = {"N/A": 0, "Pass": 1, "Low": 2, "Medium": 3, "High": 4}
order_checks = list(dict.fromkeys(findings["check"]))
pivot = pivot.loc[order_checks]
num = pivot.replace(rank).astype(float)
plt.figure(figsize=(8, 7))
sns.heatmap(num, annot=pivot, fmt="", linewidths=.5, cbar=False,
            cmap=ListedColormap(["#D9D9D9", "#54A24B", "#9ECAE9", "#F2B134", "#E45756"]), vmin=0, vmax=4)
plt.title("Check results (green = pass, grey = not applicable)"); plt.xlabel(""); plt.ylabel("")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/check_matrix.png", dpi=150); plt.show()

## 5. Scan your own sites (optional)
Add URLs you own or are allowed to test, then run this cell and the cells in sections 3-4 again. Example:
`EXTERNAL_TARGETS = {"my-site": "https://example.com/"}`

In [ ]:
EXTERNAL_TARGETS = {}      # e.g. {"my-site": "https://example.com/"}
if EXTERNAL_TARGETS:
    TARGETS.update(EXTERNAL_TARGETS)
    print("Targets:", list(TARGETS))
else:
    print("No external targets configured; scanning the local lab servers only.")

## 6. Self-check
The scanner is only useful if it reports what we know to be true about the lab servers. These checks fail loudly if the scanner and the servers disagree.

In [ ]:
sc = summary["Score"]
assert sc["insecure-lab"] < sc["partial-lab"] < sc["hardened-lab"], "scores should rise with hardening"
assert summary.loc["insecure-lab", "Grade"] == "F"
assert summary.loc["hardened-lab", "Grade"] == "A" and summary.loc["hardened-lab", "High"] == 0
f = findings.set_index(["target", "check"])["severity"]
assert f[("insecure-lab", "Reflected origin")] == "High"
assert f[("insecure-lab", "Null origin")] == "High"
assert f[("partial-lab", "Wildcard origin")] == "Low"
assert f[("hardened-lab", "Reflected origin")] == "Pass"
assert f[("hardened-lab", "Cookie HttpOnly")] == "Pass"
print("All self-checks passed.")

## 7. Generate the project report (.docx)
Builds `AmitKumar_ProjectReport.docx` from the scan results above (scores, tables, charts and discussion). Run it after sections 3-4.

In [ ]:
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH

doc = Document()
doc.styles["Normal"].font.name = "Calibri"
doc.styles["Normal"].font.size = Pt(11)

def para(text, bold=False, italic=False, align=None):
    p = doc.add_paragraph(); r = p.add_run(text); r.bold = bold; r.italic = italic
    if align: p.alignment = align
    return p
def bullets(items):
    for t in items: doc.add_paragraph(t, style="List Bullet")
def table(header, rows, widths=None):
    t = doc.add_table(rows=1, cols=len(header)); t.style = "Light Grid Accent 1"
    for i, h in enumerate(header): t.rows[0].cells[i].text = h
    for r in rows:
        cells = t.add_row().cells
        for i, v in enumerate(r): cells[i].text = str(v)
    doc.add_paragraph()
def figure(path, caption, width=5.8):
    doc.add_picture(path, width=Inches(width))
    doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
    para(caption, italic=True, align=WD_ALIGN_PARAGRAPH.CENTER)

lab = [t for t in summary.index if t.endswith("-lab")]
n_checks = findings.groupby("target").size().iloc[0]

for _ in range(5): doc.add_paragraph()
t = doc.add_heading("Web Security Scanner", 0); t.alignment = WD_ALIGN_PARAGRAPH.CENTER
para("An automated Python audit of HTTP security headers, cookies and CORS configuration", align=WD_ALIGN_PARAGRAPH.CENTER)
para("Submitted by: Amit Kumar", bold=True, align=WD_ALIGN_PARAGRAPH.CENTER)
doc.add_page_break()

doc.add_heading("1. Introduction", 1)
para("Many web attacks, such as cross-site scripting, clickjacking, session theft and cross-origin data theft, can be "
     "blocked or reduced by a handful of HTTP response headers and cookie settings. These settings are easy to forget "
     "and easy to get wrong, and checking them by hand on every site does not scale.")
para("This project is a Python scanner that sends safe, read-only requests to a web application and audits "
     "its security headers, cookie flags, information leaks and CORS (Cross-Origin Resource Sharing) policy. Each issue "
     "gets a severity and a fix, and each site gets a score and a letter grade.")
doc.add_heading("1.1 Objectives", 2)
bullets(["Check the most important security headers and report which are missing or weak.",
         "Detect common CORS mistakes: reflected origins, wildcard origins, the null origin and risky preflight methods.",
         "Check cookies for the HttpOnly, Secure and SameSite flags.",
         "Turn the findings into a score, a grade and fix recommendations.",
         "Verify the scanner against test servers whose configuration is known."])

doc.add_heading("2. What the Scanner Checks", 1)
table(["Category", "Check", "Why it matters"], [
    ["Headers", "Strict-Transport-Security", "Forces browsers to use HTTPS and stops downgrade attacks."],
    ["Headers", "Content-Security-Policy", "Limits where scripts can load from, reducing the impact of XSS."],
    ["Headers", "Clickjacking protection", "X-Frame-Options or frame-ancestors stops the page being framed by attackers."],
    ["Headers", "X-Content-Type-Options", "Stops browsers guessing content types (MIME sniffing)."],
    ["Headers", "Referrer-Policy", "Controls how much URL information leaks to other sites."],
    ["Headers", "Permissions-Policy", "Disables browser features the site does not need."],
    ["Info leak", "Server version, X-Powered-By", "Version details help attackers pick known exploits."],
    ["Cookies", "HttpOnly, SameSite, Secure", "Protect session cookies from scripts, CSRF and plain-HTTP exposure."],
    ["CORS", "Reflected origin", "A server that echoes any Origin lets any website read its responses."],
    ["CORS", "Wildcard origin", "Allows every site to read the response."],
    ["CORS", "Null origin", "Sandboxed pages can send Origin: null, so trusting it is unsafe."],
    ["CORS", "Preflight methods", "Untrusted origins should not be allowed to use PUT, DELETE or PATCH."],
])
para("Scoring: a site starts at 100 points. Each High finding costs 20 points, Medium 10 and Low 4. "
     "Grades are A (90 or more), B (80), C (65), D (50) and F (below 50).")

doc.add_heading("3. Methodology", 1)
bullets(["The scanner is written in Python using the requests library. It only sends ordinary GET and OPTIONS requests and never attempts exploitation.",
         "CORS is tested by sending requests with Origin: https://evil.example and Origin: null, plus an OPTIONS preflight asking for DELETE.",
         "Every check is recorded as Pass, Low, Medium, High or N/A so a complete pass/fail matrix can be drawn.",
         f"To make the results repeatable, the notebook starts {len(lab)} small local web servers with different configurations on purpose: an insecure one, a partially secured one and a hardened one. These give the scanner known answers to be tested against.",
         "A self-check cell asserts that the scanner reports the expected findings for each test server."])
para("Scanning real sites is supported (section 5 of the notebook) but should only be done on sites you own or have permission to test.")

doc.add_heading("4. Results", 1)
para(f"The scanner ran {n_checks} checks against each of the {len(lab)} test servers.")
table(["Target", "Score", "Grade", "High", "Medium", "Low", "Passed"],
      [[n, r["Score"], r["Grade"], r["High"], r["Medium"], r["Low"], r["Passed"]] for n, r in summary.iterrows()])
figure(f"{RESULTS_DIR}/scores.png", "Figure 1: Security score per target", 4.8)
figure(f"{RESULTS_DIR}/severity.png", "Figure 2: Findings by severity", 4.8)
figure(f"{RESULTS_DIR}/check_matrix.png", "Figure 3: Result of every check on every target", 5.6)

doc.add_heading("4.1 Findings for each target", 2)
for name in summary.index:
    g = issues[issues["target"] == name]
    doc.add_heading(f"{name} (score {summary.loc[name, 'Score']}, grade {summary.loc[name, 'Grade']})", 3)
    if g.empty:
        para("No issues found.")
    else:
        table(["Severity", "Check", "Finding", "Recommended fix"],
              [[r["severity"], r["check"], r["detail"], r["fix"]] for _, r in g.iterrows()])

doc.add_heading("5. Discussion", 1)
hi = findings[(findings["severity"] == "High")]
para(f"The scores follow the amount of hardening: {', '.join(f'{n} scored {summary.loc[n, 'Score']}' for n in summary.index)}. "
     "This matches how each test server was set up, which shows that the checks and the scoring behave sensibly.")
para(f"The most serious problems were in CORS. The insecure server echoed back any origin while also allowing credentials "
     f"and trusted the null origin, so {len(hi)} High findings were raised in total. This combination would let any website "
     "read private responses on behalf of a logged-in user, which is why it is rated higher than a missing header.")
para("The partially secured server shows a common real-world pattern: a few easy headers were added (nosniff and "
     "X-Frame-Options), but there was no Content-Security-Policy, Referrer-Policy or Permissions-Policy, the cookie "
     "had no SameSite attribute, and CORS used a wildcard. The hardened server passed every check.")
doc.add_heading("5.1 Limitations", 2)
bullets(["Only passive checks are done; the scanner does not look for vulnerabilities such as SQL injection or XSS directly.",
         "The results here come from local test servers built for this project, so they show that the scanner detects problems correctly, not how common these problems are on the internet.",
         "A site can set different headers on different pages; only the URL given is checked.",
         "The CSP check looks for obviously weak values but does not fully parse every policy.",
         "Results are a snapshot in time and should be repeated after configuration changes."])

doc.add_heading("6. Conclusion and Future Work", 1)
para("The scanner finds missing or weak security headers, cookie flags and CORS misconfigurations, explains how to fix "
     "each one, and summarises the result as a score. It correctly separated the insecure, partial and hardened test "
     "servers.")
bullets(["Crawl several pages of a site and compare their headers.",
         "Check TLS settings and certificate expiry.",
         "Add HTML and PDF report output and a command-line interface.",
         "Run the scanner in a CI/CD pipeline to catch header regressions before release."])

doc.add_heading("7. Technologies Used", 1)
para("Python, Jupyter Notebook, requests, pandas, matplotlib, seaborn, python-docx and the built-in http.server module. "
     "Setup steps are in README.md.")

doc.add_heading("8. References", 1)
bullets(["OWASP Secure Headers Project: https://owasp.org/www-project-secure-headers/",
         "MDN Web Docs, Cross-Origin Resource Sharing (CORS): https://developer.mozilla.org/en-US/docs/Web/HTTP/CORS",
         "MDN Web Docs, HTTP security headers and Content-Security-Policy: https://developer.mozilla.org/en-US/docs/Web/HTTP/CSP",
         "OWASP Web Security Testing Guide: https://owasp.org/www-project-web-security-testing-guide/"])

doc.save("AmitKumar_ProjectReport.docx")
print("Saved AmitKumar_ProjectReport.docx")